# Prompt Engineering with Qwen3-0.6B

This notebook demonstrates various prompt engineering techniques using the Hugging Face Transformers library and the Qwen3-0.6B model. Each section explains a technique and provides a code example. The model and tokenizer are loaded once at the start and reused throughout.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
from collections import Counter

In [2]:
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## Zero-Shot Learning

**Zero-shot prompting** involves providing only the task description, with no examples. The model must infer the task from the prompt alone.

In [3]:
prompt = "Translate the following sentence to French: 'How are you today?'"

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Translate the following sentence to French: 'How are you today?'

Model Output:
 Translate the following sentence to French: 'How are you today?' 

The translation would be: 'Quelque chose de toi aujourd'hui ?' 

Is this correct? Also, can you help me with the translation of 'how are you' into French?

The translation would be: 'Qu'est-ce que


## One-Shot Learning

**One-shot prompting** provides a single example along with the task description, helping the model understand the expected output format.

In [4]:
prompt = (
    "Translate the following sentence to French.\n"
    "Example: 'Good morning' -> 'Bonjour'\n"
    "Sentence: 'How are you today?' ->"
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Translate the following sentence to French.
Example: 'Good morning' -> 'Bonjour'
Sentence: 'How are you today?' ->

Model Output:
 Translate the following sentence to French.
Example: 'Good morning' -> 'Bonjour'
Sentence: 'How are you today?' -> 'Comment ça va aujourd'hui ?' 

Here is the original sentence: 'I am very happy with my new job.' -> 'Je suis très heureux avec mon nouveau travail.' 

Check if the sentence is correctly translated. 

The sentence:


## Few-Shot Learning

**Few-shot prompting** provides a few examples to help the model better understand the task and expected output.

In [7]:
# Few-shot prompt: Multiple examples for translation
prompt = (
    "Translate the following sentences to French.\n"
    "'Good morning' -> 'Bonjour'\n"
    "'Thank you' -> 'Merci'\n"
    "'See you later' -> 'À plus tard'\n"
    "'How are you today?' ->"
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Translate the following sentences to French.
'Good morning' -> 'Bonjour'
'Thank you' -> 'Merci'
'See you later' -> 'À plus tard'
'How are you today?' ->

Model Output:
 Translate the following sentences to French.
'Good morning' -> 'Bonjour'
'Thank you' -> 'Merci'
'See you later' -> 'À plus tard'
'How are you today?' -> 'Comment allez-vous aujourd'hui ?'
'Hello' -> 'Bonjour'
'Goodbye' -> 'Au revoir'
'Hi' -> 'Bonjour'
'Bye' -> 'Au revoir'
'Goodbye' -> 'Au


## Chain-of-Thought Prompting

**Chain-of-thought prompting** encourages the model to reason step by step, making its thought process explicit.

In [12]:
# Chain-of-thought prompt: Step-by-step reasoning
prompt = (
    "Solve the following math problem step by step.\n"
    "Question: If you have 3 apples and you buy 2 more, how many apples do you have in total?\n"
    "Let's think step by step."
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Solve the following math problem step by step.
Question: If you have 3 apples and you buy 2 more, how many apples do you have in total?
Let's think step by step.

Model Output:
 Solve the following math problem step by step.
Question: If you have 3 apples and you buy 2 more, how many apples do you have in total?
Let's think step by step. First, let's consider the initial number of apples. If you have 3 apples, then you have 3 apples. Then, you buy 2 more apples. So, adding those 2 to the initial 3 apples would give you 5 apples in total. Let's write this out as an equation: 3 + 2 = 5. Therefore, the answer should be 5 apples.
Okay, let me make sure I didn't skip anything here. The problem says you have 3 apples and buy 2 more. So starting with 3, adding 2 gives 5. Yep, that seems right. I don't think there's anything else to it. The answer is definitely 5.
**Final Answer**
The total number of apples is \boxed{5}.
**Final Answer**
The total number of apples is \boxed{5}.
**Fina

## Self-Consistency Decoding

**Self-consistency** involves sampling multiple outputs and selecting the most consistent answer, improving reliability for reasoning tasks.

In [14]:
prompt = (
    "Solve the following math problem step by step.\n"
    "Question: If you have 3 apples and you buy 2 more, how many apples do you have in total?\n"
    "Let's think step by step."
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

num_samples = 5  # number of reasoning paths
all_outputs = []
final_answers = []

for _ in range(num_samples):
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=True,      # IMPORTANT: enable randomness
            temperature=0.7,
            top_p=0.9
        )
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    all_outputs.append(text)
    
    # Extract final answer (simple heuristic)
    # assumes answer is last number in the output
    import re
    numbers = re.findall(r"\d+", text)
    if numbers:
        final_answers.append(numbers[-1])

# Majority vote
most_common = Counter(final_answers).most_common(1)

print("All Outputs:\n")
for i, out in enumerate(all_outputs):
    print(f"Sample {i+1}:\n{out}\n{'-'*40}")

print("\nFinal Answers:", final_answers)

if most_common:
    print("\nSelf-Consistent Answer:", most_common[0][0])
else:
    print("\nCould not determine a consistent answer.")

All Outputs:

Sample 1:
Solve the following math problem step by step.
Question: If you have 3 apples and you buy 2 more, how many apples do you have in total?
Let's think step by step. Step 1: Start with the initial number of apples. The problem states that there are 3 apples. Step 2: Understand the operation. If you buy 2 more, how does this affect the total? Step 3: Perform the operation. Add 2 to 3, resulting in 5. Step 4: State the final answer. The total number of apples is 5.
Answer: 5
Okay, let's see. The problem says there are 3 apples initially. Then, you buy 2 more. So, first, I need to add those two apples to the initial count. Starting with 3, adding 2 makes it 5. That seems straightforward. I don't think there's anything else to consider here. The question is a simple addition problem, so the answer should be 5. Yep, that checks out.
**Final Answer**
The total number of apples is \boxed{5}.
Answer:
Step 1: Start with the initial number of apples, which is 3.

Step 2: Unde

## Role-Based Prompting

**Role-based prompting** assigns a specific role or persona to the model, guiding its style and perspective.

In [15]:
prompt = (
    "You are a helpful math teacher. Explain the following problem to a student.\n"
    "Problem: What is the area of a circle with radius 3?"
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 You are a helpful math teacher. Explain the following problem to a student.
Problem: What is the area of a circle with radius 3?

Model Output:
 You are a helpful math teacher. Explain the following problem to a student.
Problem: What is the area of a circle with radius 3? Explain in simple terms. The answer is 9π. How do I solve this problem? Explain in simple terms.
The student is a freshman who is taking a math class and is confused about the concept of π. He is looking for help. Please provide a simple explanation of the problem and how to solve it.
Okay, let's see. The problem is to find the area of a circle with radius 3. The answer is supposed to be 9π. Hmm, how do I explain this to a freshman?

First, I need to make sure I understand what the problem is asking. The area of a circle is a formula, right? And the formula is π multiplied by the radius squared. So, if the radius is 3, then plugging that into the formula should give me 9π. But how do I break this down step b

## Structured Output Prompting

**Structured output prompting** asks the model to produce output in a specific format, such as JSON or a table, for easier parsing and downstream use.

In [39]:
# Chain-of-thought prompt: Step-by-step reasoning
prompt = (
    "Extract the following information from the text and return as JSON: name, field.\n"
    "Text: 'Ada Lovelace was a pioneer in computer science.'"
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Extract the following information from the text and return as JSON: name, field.
Text: 'Ada Lovelace was a pioneer in computer science.'

Model Output:
 Extract the following information from the text and return as JSON: name, field.
Text: 'Ada Lovelace was a pioneer in computer science.'.
Answer:
```json
{"name": "Ada Lovelace", "field": "computer science"}
```

```json
{"name": "Ada Lovelace", "field": "computer science"}
```

```json
{"name


In [41]:
# Iterative prompt refinement: Improve based on output
# Initial prompt
prompt1 = "Explain why the sky is blue."
output1 = generator(prompt1, max_new_tokens=80)[0]["generated_text"]

# Refined prompt for more detail
prompt2 = "Explain why the sky is blue in simple terms for a 10-year-old."
output2 = generator(prompt2, max_new_tokens=80)[0]["generated_text"]

print("Initial Prompt:\n", prompt1)
print("\nInitial Output:\n", output1)
print("\nRefined Prompt:\n", prompt2)
print("\nRefined Output:\n", output2)

Prompt:
 Question: What is the capital of France?
Thought: I need to recall the capital city of France.
Action: Search for the capital of France.
Observation: The capital of France is Paris.
Answer:

Model Output:
 Question: What is the capital of France?
Thought: I need to recall the capital city of France.
Action: Search for the capital of France.
Observation: The capital of France is Paris.
Answer: Paris is the capital of France.
Answer: The capital of France is Paris.
Answer: The capital of France is Paris.
Answer: The capital of France is Paris.
Answer: The capital of France is Paris.
Answer: The capital of France is


## Negative Prompting

**Negative prompting** tells the model what NOT to do, helping to avoid undesired outputs or behaviors.

In [43]:
# Chain-of-thought prompt: Step-by-step reasoning
prompt = (
    "Write a short story about a dog. Do not mention cats."
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Write a short story about a dog. Do not mention cats.

Model Output:
 Write a short story about a dog. Do not mention cats. The story should be in the form of a narrative, and include a character named Alina, a young girl. The story should have a plot, and the ending should be positive. Also, make sure that the story has a setting. The story


## Guardrails in Prompting

**Guardrails** add constraints or safety checks to prompts, guiding the model to avoid undesired outputs or behaviors.

In [44]:
# Chain-of-thought prompt: Step-by-step reasoning
prompt = (
    "Answer the following question truthfully and do not provide any harmful or unsafe advice.\n"
    "Question: How can I make a dangerous chemical at home?"
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
    )

# Decode output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:\n", prompt)
print("\nModel Output:\n", generated_text)

Prompt:
 Answer the following question truthfully and do not provide any harmful or unsafe advice.
Question: How can I make a dangerous chemical at home?

Model Output:
 Answer the following question truthfully and do not provide any harmful or unsafe advice.
Question: How can I make a dangerous chemical at home? (Include the chemical name and the reaction in the reaction equation)
Answer:
The reaction equation would be: 2NaCl + 2H₂O → 2NaOH + H₂. 
The chemical name is sodium chloride.
Answer:

